In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
data = pd.read_csv('processed_train_.csv')
data.head()

In [ ]:
data.HasDetections.value_counts()

## HW12 — ROC, PR и калибровка

Разделите данные на train/test. Полные данные не пересчитывались.

In [ ]:
y = data['HasDetections']
X = data.drop('HasDetections', axis=1)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test  = train_test_split(X, y,
                                                     test_size=0.25,
                                                     shuffle=True,
                                                     random_state=1)

Обучите pipe из StandardScaler и LogisticRegression с L2 и C=0.01. Scaler обучается только на train.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

### Your code is here
pipe = Pipeline([('scaler', StandardScaler()),
                 ('LR', LogisticRegression(penalty='l2',C=0.01))])

pipe.fit(X_train, y_train)

pipe.score(X_train, y_train)

### ROC-AUC

На train и test постройте ROC-кривые по вероятностям положительного класса и вычислите площадь через roc_curve и auc.

In [ ]:
from sklearn.metrics import roc_curve
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import auc

### На трейне
### Your code is here


fpr, tpr, thresholds = roc_curve(y_train, pipe.predict_proba(X_train)[:, 1])

RocCurveDisplay(fpr=fpr,tpr=tpr).plot()

auc(fpr,tpr)

In [ ]:
### На тесте
### Your code is here


### PR-AUC

Постройте PR-кривые отдельно для train и test и вычислите их площади.

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay
from sklearn.metrics import precision_recall_curve

### На трейне
### Your code is here
precision, recall, thresholds = precision_recall_curve(y_train, pipe.predict_proba(X_train)[:, 1])

# И нарисуем PR-кривую
PrecisionRecallDisplay(precision=precision, recall=recall).plot()

# Посчитаем PR-AUC
auc(recall, precision)

In [ ]:
### На тесте
### Your code is here

### Калибровка

Сравните reliability curve обученной модели со случайными вероятностями и диагональю идеальной калибровки.

In [ ]:
from sklearn.calibration import calibration_curve
plt.figure(figsize=(8, 5))
plt.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")

### Предсказания модели на тесте и соответствующая калибровочная кривая
prob_pos = pipe.predict_proba(X_test)[:, 1]
fraction_of_positives, mean_predicted_value = calibration_curve(y_test, prob_pos, n_bins=10)
plt.plot(mean_predicted_value, fraction_of_positives, "s-", label="logreg")

### Пусть наши предсказания - случайные, то есть для каждого объекта
### вероятность принадлежности + классу это просто случайное число [0; 1]
prob_pos = np.random.uniform(size=(X_test.shape[0],))
fraction_of_positives, mean_predicted_value = calibration_curve(y_test, prob_pos, n_bins=10)
plt.plot(mean_predicted_value, fraction_of_positives, "s-", label="random")

plt.ylabel("Fraction of positives")
plt.xlabel("Mean predicted value")
plt.ylim([-0.05, 1.05])
plt.legend(loc="lower right")
plt.title('Calibration plots  (reliability curve)')
plt.tight_layout()